# Volume 3, Chapter 5: Continuous Probability Done Properly (Python companion)

Exercises first, solutions at the end. A computer only produces (pseudo-)uniform numbers; everything else is built from them. We turn uniforms into exponential, Cauchy and Gaussian variables in three ways, watch the Cauchy law break the law of large numbers, use Monte Carlo to compute integrals and volumes in high dimension, and sample the Cantor distribution, which has neither atoms nor a density.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(5)
Phi = lambda t: 0.5 * (1 + math.erf(t / math.sqrt(2)))   # N(0,1) distribution function

## Exercises

**P5.1 (★) The inverse transform.** Using Proposition 5.18, sample $10^6$ exponential$(2)$ variables, $10^6$ Cauchy variables and $10^6$ draws from the law $P(X=k)=2^{-k}$ ($k\ge1$) from uniform numbers. Measure $\sup_t|F_n(t)-F(t)|$, where $F_n$ is the empirical distribution function.

**P5.2 (★) Box–Muller.** Implement Theorem 5.27. Check the means, variances and correlation of $X$, $Y$, compare $P(X\le t)$ with $\Phi(t)$, and check that $X^2+Y^2$ is exponential with mean $2$ (Exercise 5.4).

**P5.3 (★★) Two more Gaussian samplers.** (a) *Rejection*: propose $Z$ with the Laplace density $\frac12e^{-|x|}$ and accept it with probability $\frac{\varphi(Z)}{M\cdot\frac12e^{-|Z|}}$, where $M=\sup_x\frac{\varphi(x)}{\frac12e^{-|x|}}$. Compute $M$, the acceptance rate, and check the output. (b) *Marsaglia's polar method*: draw $(V_1,V_2)$ uniform in $[-1,1]^2$ until $S=V_1^2+V_2^2\in\,]0,1[$, and output $V_i\sqrt{-2\ln S/S}$. Why does it work, and why is it faster than Box–Muller?

**P5.4 (★★) Cauchy against the law of large numbers.** Plot the running means $\frac{S_n}n$ of Cauchy and of Gaussian samples up to $n=10^6$. Then plot running *medians*. What do you see, and why?

**P5.5 (★) Order statistics.** For $n=9$ uniform points, simulate $U_{(k)}$ for $k=1,5,9$ and compare the histograms with the densities of Exercise 5.6.

**P5.6 (★★) Monte Carlo volumes and the curse of dimension.** Estimate the volume of the unit ball of $\R^d$ for $d=2,\dots,12$ by sampling the cube $[-1,1]^d$, with a $95\%$ error bar from Chebyshev or Hoeffding. Compare with $\frac{\pi^{d/2}}{\Gamma(d/2+1)}$. What happens to the relative error as $d$ grows?

**P5.7 (★★) The Cantor distribution.** Sample $X=\sum_{k\le40}2B_k3^{-k}$ (Exercise 5.7) and plot its empirical distribution function. Compute $\mathbb E[X]$ and $\operatorname{Var}X$ exactly and compare.

**P5.8 (★) Binary digits of a uniform number.** Extract the first $30$ binary digits of $10^6$ uniform floating-point numbers and test Proposition 5.23: each digit should be a fair bit, and each pattern of three consecutive digits should have frequency $\frac18$. What happens after digit $53$?

## Solutions

**P5.1.** $G$ is explicit in each case: $-\frac12\ln(1-U)$, $\tan(\pi(U-\frac12))$, and $\lceil\log_2\frac1{1-U}\rceil$ for the geometric law, since $F(k)=1-2^{-k}$ and $G(u)$ is the least $k$ with $F(k)\ge u$.

In [ ]:
n = 10**6
U = rng.random(n)
samples = {
    'exponential(2)': (-0.5 * np.log1p(-U), lambda t: 1 - np.exp(-2 * np.maximum(t, 0))),
    'Cauchy': (np.tan(np.pi * (U - 0.5)), lambda t: 0.5 + np.arctan(t) / np.pi),
    'geometric(1/2)': (np.ceil(-np.log2(1 - U)), lambda t: 1 - 2.0**-np.floor(np.maximum(t, 0))),
}
for name, (x, F) in samples.items():
    xs = np.sort(x)
    if name.startswith('geometric'):
        ks = np.arange(1, 40)
        D = np.max(np.abs(np.searchsorted(xs, ks, side='right') / n - F(ks)))   # F_n and F only jump at integers
    else:
        D = max(np.max(np.abs(np.arange(1, n + 1) / n - F(xs))),             # just after each sample point
                np.max(np.abs(np.arange(0, n) / n - F(xs))))                 # and just before (F is continuous)
    print(f'{name:15s} sup |F_n - F| = {D:.5f}   (1/sqrt(n) = {1/math.sqrt(n):.4f})')

All three distances are of order $\frac1{\sqrt n}$, as they should be: the Glivenko–Cantelli theorem says that $\sup_t|F_n-F|\to0$ almost surely, and Kolmogorov showed that $\sqrt n\sup_t|F_n-F|$ has a limit law that does not depend on $F$ (when $F$ is continuous). This is the basis of the Kolmogorov–Smirnov test that random number generators are checked with.

**P5.2.**

In [ ]:
def box_muller(m):
    U1, U2 = rng.random(m), rng.random(m)
    U1 = 1 - U1                                  # in ]0,1], so the log is finite
    R, T = np.sqrt(-2 * np.log(U1)), 2 * np.pi * U2
    return R * np.cos(T), R * np.sin(T)
X, Y = box_muller(10**6)
print('means', X.mean(), Y.mean(), ' variances', X.var(), Y.var(), ' correlation', np.corrcoef(X, Y)[0, 1])
for t in [-2, -1, 0, 0.5, 1.96]:
    print(f'P(X <= {t:5}) = {np.mean(X <= t):.4f}   Phi = {Phi(t):.4f}')
S = X**2 + Y**2
for t in [1, 2, 4, 8]:
    print(f'P(X^2+Y^2 > {t}) = {np.mean(S > t):.4f}   e^(-t/2) = {math.exp(-t/2):.4f}')

**P5.3.** (a) $\frac{\varphi(x)}{\frac12e^{-|x|}}=\sqrt{\frac2\pi}e^{|x|-x^2/2}$ is largest at $|x|=1$, so $M=\sqrt{2e/\pi}\approx1.315$. Each proposal is accepted with probability $\frac1M\approx0.760$, and the accepted values have density $\varphi$: $P(Z\in\mathrm dx,\text{accept})=g(x)\frac{\varphi(x)}{Mg(x)}\mathrm dx=\frac{\varphi(x)}M\mathrm dx$.

In [ ]:
M = math.sqrt(2 * math.e / math.pi)
m = 10**6
Z = rng.exponential(1.0, m) * rng.choice([-1, 1], m)        # Laplace proposals
accept = rng.random(m) < np.exp(-Z**2 / 2) / math.sqrt(2 * math.pi) / (M * 0.5 * np.exp(-np.abs(Z)))
G = Z[accept]
print(f'M = {M:.4f}, acceptance rate {accept.mean():.4f} (theory {1/M:.4f})')
print('mean', G.mean(), 'variance', G.var(), ' P(G <= 1) =', np.mean(G <= 1), ' Phi(1) =', Phi(1))
# (b) Marsaglia's polar method
V = rng.uniform(-1, 1, (m, 2))
S = (V**2).sum(axis=1)
ok = (S > 0) & (S < 1)
V, S = V[ok], S[ok]
P = V * np.sqrt(-2 * np.log(S) / S)[:, None]
print(f'polar method: acceptance {ok.mean():.4f} (pi/4 = {math.pi/4:.4f}); mean {P.mean():.4f}, var {P.var():.4f}, corr {np.corrcoef(P.T)[0,1]:.4f}')

(b) Given acceptance, $(V_1,V_2)$ is uniform in the unit disc, so in polar form its angle is uniform, $S$ is uniform on $]0,1[$ (the disc of radius $\sqrt s$ has a fraction $s$ of the area), and the two are independent. So $(V_1,V_2)/\sqrt S$ is $(\cos\Theta,\sin\Theta)$, and $\sqrt{-2\ln S}$ plays the role of $R$ in Box–Muller with $U_1=S$. It avoids computing $\cos$ and $\sin$, at the price of rejecting a fraction $1-\frac\pi4\approx21\%$ of the pairs.

**P5.4.**

In [ ]:
n = 10**6
C = np.tan(np.pi * (rng.random(n) - 0.5))
N = rng.standard_normal(n)
k = np.arange(1, n + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].semilogx(k, np.cumsum(C) / k, label='Cauchy'); axes[0].semilogx(k, np.cumsum(N) / k, label='Gaussian')
axes[0].set_title('running mean'); axes[0].legend()
idx = np.unique(np.logspace(0, 6, 300).astype(int))
axes[1].semilogx(idx, [np.median(C[:j]) for j in idx], label='Cauchy')
axes[1].semilogx(idx, [np.median(N[:j]) for j in idx], label='Gaussian')
axes[1].set_title('running median'); axes[1].legend(); plt.show()
print('Cauchy mean of the first 10^6:', C.mean(), '  median:', np.median(C))

The Gaussian running mean settles at $0$; the Cauchy one keeps jumping, because now and then a single huge value moves the whole average. In fact $\frac{S_n}n$ is again Cauchy for every $n$ (Chapter 6 proves it with characteristic functions), so averaging does not help at all: the law of large numbers needs $\mathbb E|X|<\infty$ (Exercise 5.4). The median, which ignores the size of extreme values, converges to $0$ in both cases.

**P5.5.**

In [ ]:
n = 9
U = np.sort(rng.random((200_000, n)), axis=1)
t = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
for k in [1, 5, 9]:
    ax.hist(U[:, k - 1], bins=80, density=True, alpha=0.35)
    ax.plot(t, n * math.comb(n - 1, k - 1) * t**(k - 1) * (1 - t)**(n - k), label=f'k = {k}')
    print(f'k = {k}: simulated mean {U[:, k-1].mean():.4f}, k/(n+1) = {k/(n+1):.4f}')
ax.legend(); plt.show()

**P5.6.** The fraction $\hat p$ of points of the cube that fall in the ball estimates $\frac{V_d}{2^d}$. Hoeffding (Theorem 4.9) gives $P(|\hat p-p|\ge t)\le2e^{-2mt^2}$, so $t=\sqrt{\ln(40)/(2m)}$ gives a $95\%$ guarantee.

In [ ]:
m = 2 * 10**6
t = math.sqrt(math.log(40) / (2 * m))
for d in range(2, 13):
    hits = 0
    for _ in range(4):                                   # in chunks, to save memory
        P = rng.uniform(-1, 1, (m // 4, d))
        hits += np.count_nonzero((P**2).sum(axis=1) <= 1)
    p = hits / m
    exact = math.pi**(d / 2) / math.gamma(d / 2 + 1)
    print(f'd = {d:2d}: estimate {2**d * p:8.4f} +- {2**d * t:.4f},  exact {exact:.4f},  fraction of the cube {exact / 2**d:.2e}')

The absolute error bar $2^dt$ grows with $d$ while the volume $V_d$ goes to $0$ (it is largest at $d=5$): in dimension $12$ only one point in $3000$ falls in the ball, and the relative error is large. Uniform sampling of a box wastes almost all its points in high dimension; this *curse of dimensionality* is why Markov chain Monte Carlo (Chapter 8) samples where the mass is instead.

**P5.7.** $\mathbb E[X]=\sum_k2\cdot\frac12\cdot3^{-k}=\frac12$ and, the digits being independent, $\operatorname{Var}X=\sum_k4\cdot\frac14\cdot9^{-k}=\frac18$.

In [ ]:
m = 200_000
B = rng.integers(0, 2, (m, 40))
X = (2 * B * 3.0**-np.arange(1, 41)).sum(axis=1)
print('mean', X.mean(), '(1/2)   variance', X.var(), '(1/8 =', 1/8, ')')
xs = np.sort(X)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(xs, np.arange(1, m + 1) / m)
ax.set_title('the Cantor staircase (empirical distribution function)'); plt.show()
print('fraction of samples in the middle third ]1/3, 2/3[:', np.mean((X > 1/3) & (X < 2/3)))

The staircase is flat on every removed third, rises only on the Cantor set, and still climbs continuously from $0$ to $1$: no atoms, no density. Such *singular continuous* laws are the third kind in Lebesgue's decomposition of any law (Volume 6), after discrete laws and laws with a density.

**P5.8.** `numpy` floats in $[0,1[$ are multiples of $2^{-53}$: the digits beyond the $53$rd are all $0$.

In [ ]:
U = rng.random(10**6)
digits = np.zeros((len(U), 60), dtype=np.int8)
x = U.copy()
for k in range(60):
    x = 2 * x
    digits[:, k] = (x >= 1)
    x -= digits[:, k]
means = digits.mean(axis=0)
print('frequency of 1 for digits 1..10:', np.round(means[:10], 4))
print('frequency of 1 for digits 50..60:', np.round(means[49:60], 4))
pat = digits[:, 0] * 4 + digits[:, 1] * 2 + digits[:, 2]
print('patterns of digits 1-3:', np.round(np.bincount(pat, minlength=8) / len(U), 4))
pat = digits[:, 20] * 4 + digits[:, 21] * 2 + digits[:, 22]
print('patterns of digits 21-23:', np.round(np.bincount(pat, minlength=8) / len(U), 4))

Up to digit $53$ everything looks like independent fair bits; after it, the digits are $0$. A computer's ``uniform'' number is a uniform choice among $2^{53}$ dyadic numbers, which is exactly what the first $53$ coins of Proposition 5.23 give. For almost all purposes this is indistinguishable from the real thing, but a simulation of a rare event of probability $10^{-17}$ would need more care.